Import the dataset and setup git

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/amanganglani/.cache/kagglehub/datasets/clmentbisaillon/fake-and-real-news-dataset/versions/1


In [6]:
import os

# List the contents of the downloaded dataset directory
dataset_files = os.listdir(path)
print(f"Files in the dataset directory '{path}':\n{dataset_files}")

Files in the dataset directory '/Users/amanganglani/.cache/kagglehub/datasets/clmentbisaillon/fake-and-real-news-dataset/versions/1':
['Fake.csv', 'True.csv']


Visualize the dataset

In [7]:
import numpy as np
import pandas as pd

np.random.seed(42)

true_news_df = pd.read_csv(os.path.join(path, 'True.csv'))
fake_news_df = pd.read_csv(os.path.join(path, 'Fake.csv'))

print("First 5 rows of True News dataset:")
display(true_news_df.head())

print("\nFirst 5 rows of Fake News dataset:")
display(fake_news_df.head())

First 5 rows of True News dataset:


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"



First 5 rows of Fake News dataset:


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [8]:
# Add 'truth' column to true_news_df with value 1
true_news_df['truth'] = 1

# Add 'truth' column to fake_news_df with value 0
fake_news_df['truth'] = 0

# Combine the dataframes
combined_df = pd.concat([true_news_df, fake_news_df], ignore_index=True)

print("Combined DataFrame head:")
display(combined_df.head())

print("\nCombined DataFrame tail:")
display(combined_df.tail())

print(f"\nShape of combined DataFrame: {combined_df.shape}")
print(f"Number of true news entries: {combined_df[combined_df['truth'] == 1].shape[0]}")
print(f"Number of fake news entries: {combined_df[combined_df['truth'] == 0].shape[0]}")

Combined DataFrame head:


,title,text,subject,date,truth
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1



Combined DataFrame tail:


,title,text,subject,date,truth
44893,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",0
44894,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",0
44895,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",0
44896,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",0
44897,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",0



Shape of combined DataFrame: (44898, 5)
Number of true news entries: 21417
Number of fake news entries: 23481


Randomize dataset, split into train/test 

In [10]:
# 1. Randomize combined_df so rows are in random order
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 2. Create master with only text and truth columns
master = combined_df[["text", "truth"]].copy()

# 3. 70% train / 30% test split
from sklearn.model_selection import train_test_split

train, test = train_test_split(master, test_size=0.3, random_state=42)

print(f"Master shape: {master.shape}")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Master shape: (44898, 2)
Train shape: (31428, 2)
Test shape: (13470, 2)


## Fine-Tuning RoBERTa for Fake News Detection

We fine-tune a pretrained RoBERTa transformer on the training set and evaluate on the held-out test set. Following the same approach as the BERT model example, we use a very small learning rate (1e-5) to gently update the pretrained weights without destroying what the model already learned.

In [ ]:
pip install --quiet keras-hub

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

import keras
import keras_hub

keras.utils.set_random_seed(42)

### Load the Pretrained RoBERTa Classifier

We load `roberta_base_en` (~124M parameters) from Keras Hub and add a 2-class classification head on top (fake=0, real=1). All backbone weights are set to trainable so the whole model adapts to our task.

In [ ]:
classifier = keras_hub.models.RobertaClassifier.from_preset(
    "roberta_base_en",
    num_classes=2,
)

classifier.backbone.trainable = True
classifier.summary()

### Compile the Model

We use `SparseCategoricalCrossentropy` because our labels are integers (0 or 1). A very small learning rate (1e-5) is critical — too large and we would overwrite the rich pretrained representations.

In [ ]:
classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(1e-5),
    metrics=["accuracy"],
    jit_compile=True,
)

### Prepare Text and Labels

Extract the `text` column and `truth` labels as plain Python lists. `keras-hub` preprocessors accept raw strings directly, so no manual tokenization is needed.

In [ ]:
train_texts  = train["text"].tolist()
train_labels = train["truth"].tolist()

test_texts  = test["text"].tolist()
test_labels = test["truth"].tolist()

print(f"Training samples: {len(train_texts)}")
print(f"Test samples:     {len(test_texts)}")

### Train the Model

Fine-tune for 5 epochs using 10% of the training data as a validation split to monitor for overfitting. With 31k training samples this will be slow on CPU — use a GPU runtime if available.

In [ ]:
history = classifier.fit(
    x=train_texts,
    y=train_labels,
    batch_size=16,
    epochs=5,
    validation_split=0.1,
)

### Visualize Training Dynamics

Plot loss and accuracy curves to see how the model improved across epochs and whether it is overfitting.

In [ ]:
import matplotlib.pyplot as plt

train_loss = history.history["loss"]
val_loss   = history.history["val_loss"]
train_acc  = history.history["accuracy"]
val_acc    = history.history["val_accuracy"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_loss, label="Train Loss")
axes[0].plot(val_loss,   label="Val Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].set_title("Training & Validation Loss")

axes[1].plot(train_acc, label="Train Accuracy")
axes[1].plot(val_acc,   label="Val Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].set_title("Training & Validation Accuracy")

plt.tight_layout()
plt.show()

### Evaluate on the Test Set

Run the final trained model against the held-out test set to get an unbiased accuracy estimate.

In [ ]:
test_loss, test_acc = classifier.evaluate(x=test_texts, y=test_labels)

print(f"\nTest Accuracy:  {test_acc:.4f}")
print(f"Test Loss:      {test_loss:.4f}")